In [ ]:
! pip install azure-ai-formrecognizer azure-storage-blob azure-search-documents openai

In [ ]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.storage.blob import BlobClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType
import openai
import uuid
import textwrap
from dotenv import load_dotenv

load_dotenv()
# Set your Azure credentials
AZURE_FORM_RECOGNIZER_ENDPOINT = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
AZURE_FORM_RECOGNIZER_KEY = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY")

AZURE_BLOB_CONN_STR = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_KEY = os.getenv("AZURE_SEARCH_ADMIN_KEY")
AZURE_SEARCH_INDEX = "docs-index"

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = "gpt-4"

In [ ]:
print(AZURE_BLOB_CONN_STR)

In [2]:


# 1. Upload document to Blob Storage
def upload_to_blob(local_path, container_name, blob_name):
    blob_client = BlobClient.from_connection_string(AZURE_BLOB_CONN_STR, container_name, blob_name)
    with open(local_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)
    return blob_client.url

# 2. Analyze document
def extract_text_from_document(blob_url):
    doc_client = DocumentAnalysisClient(endpoint=AZURE_FORM_RECOGNIZER_ENDPOINT,
                                        credential=AzureKeyCredential(AZURE_FORM_RECOGNIZER_KEY))
    poller = doc_client.begin_analyze_document_from_url("prebuilt-document", document_url=blob_url)
    result = poller.result()

    full_text = "\n".join([p.content for page in result.pages for p in page.lines])
    return full_text

# 3. Chunk content
def chunk_text(text, max_length=1000):
    return textwrap.wrap(text, width=max_length)

# 4. Create or use a search index
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchableField, SearchFieldDataType

def create_search_index():
    index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT,
                                     credential=AzureKeyCredential(AZURE_SEARCH_KEY))
    
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
    ]
    
    index = SearchIndex(name=AZURE_SEARCH_INDEX, fields=fields)
    
    try:
        index_client.create_index(index)
    except Exception as e:
        print(f"Index may already exist or another error occurred: {e}")


# 5. Index chunks
def index_chunks(chunks):
    search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT,
                                 index_name=AZURE_SEARCH_INDEX,
                                 credential=AzureKeyCredential(AZURE_SEARCH_KEY))
    docs = [{"id": str(uuid.uuid4()), "content": chunk} for chunk in chunks]
    search_client.upload_documents(documents=docs)

# 6. RAG: Query -> Retrieve -> Generate
# def generate_answer(query):
#     search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT,
#                                  index_name=AZURE_SEARCH_INDEX,
#                                  credential=AzureKeyCredential(AZURE_SEARCH_KEY))
#     results = search_client.search(query)
#     context = "\n\n".join([doc["content"] for doc in results])

#     prompt = f"""You are a helpful assistant. Use the following context to answer the question:
    
#     Context:
#     {context}

#     Question: {query}
#     """

#     openai.api_type = "azure"
#     openai.api_base = AZURE_OPENAI_ENDPOINT
#     openai.api_key = AZURE_OPENAI_KEY
#     openai.api_version = "2023-07-01-preview"

#     response = openai.ChatCompletion.create(
#         engine=AZURE_OPENAI_DEPLOYMENT,
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0.3,
#     )
#     return response.choices[0].message["content"]





In [6]:
from openai import AzureOpenAI

def generate_answer(query):
    # Step 1: Retrieve context from Azure Search
    search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT,
                                 index_name=AZURE_SEARCH_INDEX,
                                 credential=AzureKeyCredential(AZURE_SEARCH_KEY))
    try:
        results = search_client.search(query)
        context = "\n\n".join([doc["content"] for doc in results])
    except Exception as e:
        return f"Search failed: {e}"

    if not context.strip():
        return "No relevant context found to answer the question."

    # Step 2: Construct prompt
    prompt = f"""You are a helpful assistant. Use the following context to answer the question:

Context:
{context}

Question: {query}
"""

    # Step 3: Use new Azure OpenAI SDK
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_KEY,
        api_version="2023-07-01-preview",
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )

    try:
        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"OpenAI call failed: {e}"


In [4]:
# Step 1: Upload document
blob_url = upload_to_blob("test1.jpg", "data-container1", "test1.jpg")

# Step 2: Extract text
text = extract_text_from_document(blob_url)

# Step 3: Chunk
chunks = chunk_text(text)

# Step 4: Setup index
create_search_index()

# Step 5: Index content
index_chunks(chunks)



In [7]:
# Step 6: Ask a question
query = "What is the invoice total?"
answer = generate_answer(query)
print(f"\nAnswer: {answer}")


Answer: The invoice total is $7350.00.
